# Oturum 4 — Martini 3 Girdi Hazırlama

**Biyofizik 2026 Kursu · 25 Ağustos 2026 · Dr. Öğr. Üyesi Ekrem Yaşar**

Sabah oturumlarında CHARMM-GUI arayüzü ile atomistik olarak hazırlanan protein
(`6JOD` A zinciri, anjiyotensin II tip-2 reseptörü), bu not defterinde komut
satırı araçlarıyla ve Martini 3 kaba-taneli modeli kullanılarak yeniden
hazırlanmaktadır.

Kursta üretim simülasyonu koşulmamaktadır; amaç simülasyona girecek sistemin
kurulmasıdır.

**İşlem sırası**

1. Yazılım kurulumu
2. Yapının hazırlanması (A zincirinin ayıklanması)
3. `martinize2` ile kaba-taneli modele dönüştürme
4. Martini 3 kuvvet alanı dosyalarının indirilmesi
5. `insane` ile membran, çözücü ve iyon eklenmesi
6. Topoloji dosyasının düzeltilmesi
7. `gmx grompp` ile doğrulama
8. İsteğe bağlı enerji minimizasyonu


---
## 1. Yazılım kurulumu

Aşağıdaki hücrenin çalışma süresi yaklaşık 3–5 dakikadır ve oturum başına bir
kez çalıştırılması yeterlidir.

| Paket | İşlevi |
|---|---|
| `vermouth` | `martinize2` komutunu sağlamaktadır |
| `insane` | Membran ve kutu inşası |
| `gromacs` | `gmx grompp` ile doğrulama |
| `dssp` | İkincil yapı tayini (`mkdssp`) |


In [ ]:
%%capture
!pip install -q vermouth insane
!apt-get -qq update
!apt-get -qq install -y gromacs dssp


In [ ]:
# Kurulumun dogrulanmasi
!martinize2 --version 2>&1 | head -2
!insane --help 2>&1 | head -3
!gmx --version 2>&1 | grep -i 'GROMACS version'
!which mkdssp || echo 'mkdssp bulunamadi; asagida -ss secenegi kullanilacaktir'


---
## 2. Yapının hazırlanması

`6JOD` yapısı indirilerek yalnızca A zinciri (AT2R) ayıklanmaktadır.

Yapıda yer alan C zinciri BRIL füzyonunu, H ve L zincirleri ise Fab fragmanını
temsil etmektedir. Bu bileşenler kristalizasyon amacıyla eklenmiş deneysel
araçlar olup fizyolojik ortamda bulunmamaktadır; sisteme dâhil edilmemektedir.

Bu oturumda ligant (B zinciri) da alınmamakta, yalnızca protein ve membrandan
oluşan sade bir sistem kurulmaktadır.


In [ ]:
!wget -q https://files.rcsb.org/download/6JOD.pdb -O 6jod.pdb

# Yalnizca A zincirinin ATOM kayitlari (su ve hetero gruplar haric)
kept = []
for line in open('6jod.pdb'):
    if line.startswith('ATOM  ') and line[21] == 'A':
        kept.append(line)
kept.append('END\n')
open('at2r.pdb','w').writelines(kept)

resids = sorted({int(l[22:26]) for l in kept if l.startswith('ATOM')})
print(f'A zinciri: {len(resids)} rezidu ({resids[0]}-{resids[-1]}), {len(kept)-1} atom')


**Beklenen sonuç.** 35–340 aralığında 306 rezidü.
Dizide 312 rezidü bulunmakta olup 341–346 aralığındaki C-terminal uzantı
çözülmemiştir. Zincir içi kopukluk bulunmadığından ilmik modellemesine gerek
duyulmamaktadır.


---
## 3. `martinize2` ile kaba-taneli modele dönüştürme

Atomistik protein Martini 3 etkileşim merkezlerine dönüştürülmektedir.

| Parametre | İşlevi | Önemi |
|---|---|---|
| `-ff martini3001` | Martini 3 kuvvet alanı | Martini 2 ile karıştırılmamalıdır |
| `-dssp` | İkincil yapının belirlenmesi | Merkez tipleri ikincil yapıya bağlıdır |
| `-elastic` | Elastik ağ tanımlanması | Uygulanmadığında yapısal bütünlük korunamamaktadır |
| `-ef 700` | Yay kuvvet sabiti (kJ mol⁻¹ nm⁻²) | Yüksek değer aşırı rijitlik, düşük değer yapısal bozulma |
| `-el 0.5 -eu 0.9` | Yay mesafe aralığı (nm) | Bağlanacak merkez çiftlerini belirlemektedir |


In [ ]:
!martinize2 \
  -f at2r.pdb \
  -o topol.top \
  -x at2r_cg.pdb \
  -ff martini3001 \
  -dssp \
  -elastic -ef 700 -el 0.5 -eu 0.9 -ea 0 -ep 0 \
  -maxwarn 10


### `-dssp` seçeneğinin çalışmaması hâlinde

`mkdssp` bulunamadığında ikincil yapı doğrudan tanımlanabilmektedir. Aşağıdaki
hücre yalnızca önceki hücrenin hata vermesi durumunda çalıştırılmalıdır; tüm
diziyi heliks olarak atamakta olup bir GPCR için yaklaşık ancak kabul edilebilir
bir çözümdür.

Yayımlanacak çalışmalarda `mkdssp` kurulumunun yapılması önerilmektedir.


In [ ]:
# Yalnizca onceki hucre -dssp nedeniyle hata verdiginde calistirilmalidir
# !martinize2 -f at2r.pdb -o topol.top -x at2r_cg.pdb -ff martini3001 \
#   -ss $(python3 -c "print('H'*306)") \
#   -elastic -ef 700 -el 0.5 -eu 0.9 -ea 0 -ep 0 -maxwarn 10


In [ ]:
# Indirgeme oraninin hesaplanmasi
aa = sum(1 for l in open('at2r.pdb') if l.startswith('ATOM'))
cg = sum(1 for l in open('at2r_cg.pdb') if l.startswith(('ATOM','HETATM')))
print(f'Atomistik model : {aa:>6} atom')
print(f'Kaba-taneli model: {cg:>6} etkilesim merkezi')
print(f'Indirgeme orani : {aa/cg:.1f}')

print('\nUretilen dosyalar:')
!ls -la *.itp topol.top at2r_cg.pdb


**Elastik ağın gerekliliği.** Martini kuvvet alanı, etkileşim merkezleri
arasındaki potansiyeller aracılığıyla proteinin üçüncül yapısını koruyamamaktadır.
Yapı, harmonik yaylardan oluşan bir ağ ile kısıtlanmaktadır.

Bu yaklaşımın sonucu olarak model protein katlanma, açılma ve büyük ölçekli
konformasyonel değişim gösterememektedir. Dolayısıyla Martini modeli ile bir
GPCR'ın aktivasyon geçişi incelenememektedir. İncelenebilecek süreçler lipit
etkileşimleri, oligomerizasyon ve difüzyondur.


In [ ]:
# Uretilen topolojinin incelenmesi
import glob
itp = sorted(glob.glob('molecule_*.itp'))[0]
lines = open(itp).read().splitlines()
print('Dosya:', itp, '|', len(lines), 'satir\n')
for i, l in enumerate(lines):
    if l.strip().startswith('['):
        print(f'{i:>6}  {l.strip()}')


---
## 4. Martini 3 kuvvet alanı dosyalarının indirilmesi

`martinize2` proteinin topolojisini üretmiştir. Ayrıca Martini kuvvet alanının
genel parametre dosyaları (etkileşim merkezi tanımları, lipitler, çözücü ve
iyonlar) gerekmektedir.

Kaynak: [marrink-lab/martini-forcefields](https://github.com/marrink-lab/martini-forcefields)

Ana parametre dosyasının boyutu yaklaşık 16 MB olup indirme süresi bir dakikayı
bulabilmektedir.


In [ ]:
BASE = 'https://raw.githubusercontent.com/marrink-lab/martini-forcefields/main/martini_forcefields/regular/v3.0.0/gmx_files'
files = [
    'martini_v3.0.0.itp',
    'martini_v3.0.0_solvents_v1.itp',
    'martini_v3.0.0_ions_v1.itp',
    'martini_v3.0.0_phospholipids_v1.itp',
]
for f in files:
    !wget -q {BASE}/{f} -O {f}
!ls -lh martini_v3.0.0*.itp


---
## 5. `insane` ile membran, çözücü ve iyon eklenmesi

| Parametre | İşlevi |
|---|---|
| `-box 12,12,14` | Kutu boyutları (nm) |
| `-l POPC:1` | Lipit bileşimi |
| `-sol W` | Martini standart su modeli (bir merkez yaklaşık dört su molekülü) |
| `-salt 0.15` | 0.15 M NaCl |
| `-center` | Proteinin kutu merkezine yerleştirilmesi |

Çok bileşenli membran için: `-l POPC:7 -l POPE:2 -l CHOL:1`


In [ ]:
!insane \
  -f at2r_cg.pdb \
  -o sistem.gro \
  -p sistem_insane.top \
  -pbc square \
  -box 12,12,14 \
  -l POPC:1 \
  -sol W \
  -salt 0.15 \
  -center \
  -dm 0


In [ ]:
print('--- insane tarafindan uretilen topoloji ---')
print(open('sistem_insane.top').read())

n = int(open('sistem.gro').read().splitlines()[1])
print(f'Toplam parcacik sayisi: {n:,}')


**Değerlendirme.** Aynı hacimdeki atomistik bir sistem yaklaşık on kat daha
fazla parçacık içerecekti. Buna ek olarak Martini modeli daha büyük zaman adımı
kullanılmasına olanak vermektedir.


---
## 6. Topoloji dosyasının düzeltilmesi

Bu adım, iş akışında en sık hata alınan noktadır.

`insane` bir topoloji dosyası üretmekte, ancak `#include` yönergeleri eksik veya
hatalı olmaktadır; araç proteinin topolojisini tanımamaktadır. Dosya aşağıdaki
hücrede yeniden oluşturulmaktadır.

Yönerge sırası önemlidir: önce genel kuvvet alanı tanımları, ardından molekül
topolojileri yer almalıdır.


In [ ]:
import re, glob

prot_itp = sorted(glob.glob('molecule_*.itp'))[0]

# insane tarafindan uretilen [ molecules ] bolumu
raw = open('sistem_insane.top').read()
mols = raw.split('[ molecules ]')[1].strip().splitlines()
mols = [m for m in mols if m.strip() and not m.strip().startswith(';')]

# Protein molekul adinin martinize2 ciktisiyla eslestirilmesi
prot_name = None
for l in open(prot_itp):
    if l.strip().startswith('[ moleculetype ]'):
        continue
    if l.strip() and not l.strip().startswith((';','[')) and prot_name is None:
        prot_name = l.split()[0]
        break
print('Protein molekul adi:', prot_name)

fixed = []
for m in mols:
    parts = m.split()
    if parts[0].lower().startswith('protein'):
        fixed.append(f'{prot_name}   {parts[1]}')
    else:
        fixed.append(m.strip())

top = f'''; Biyofizik 2026 Kursu - Oturum 4
; AT2R (6JOD A zinciri) - Martini 3 - POPC membran

#include "martini_v3.0.0.itp"
#include "martini_v3.0.0_solvents_v1.itp"
#include "martini_v3.0.0_ions_v1.itp"
#include "martini_v3.0.0_phospholipids_v1.itp"
#include "{prot_itp}"

[ system ]
AT2R in POPC membrane (Martini 3)

[ molecules ]
''' + '\n'.join(fixed) + '\n'

open('sistem.top','w').write(top)
print('\n--- sistem.top ---')
print(top)


---
## 7. `gmx grompp` ile doğrulama

Bu adımda simülasyon yürütülmemektedir. Koordinat, topoloji ve parametre
dosyalarının birbiriyle tutarlılığı sınanmaktadır.

`.tpr` dosyasının üretilebilmesi, hazırlanan girdinin geçerli olduğunu
göstermektedir. Oturumun hedefi bu doğrulamanın sağlanmasıdır.


In [ ]:
mdp = '''; Martini 3 - enerji minimizasyonu
integrator               = steep
nsteps                   = 500
emtol                    = 100
emstep                   = 0.01

nstlist                  = 20
cutoff-scheme            = Verlet
verlet-buffer-tolerance  = 0.005

coulombtype              = reaction-field
rcoulomb                 = 1.1
epsilon_r                = 15
vdw_type                 = cutoff
vdw-modifier             = Potential-shift-verlet
rvdw                     = 1.1
'''
open('minimization.mdp','w').write(mdp)

!gmx grompp -f minimization.mdp -c sistem.gro -p sistem.top -o em.tpr -maxwarn 5


In [ ]:
import os
if os.path.exists('em.tpr'):
    print('em.tpr uretildi:', os.path.getsize('em.tpr'), 'bayt')
    print('Girdi hazirligi tamamlanmistir; bu dosya ile simulasyon yurutulebilir.')
else:
    print('em.tpr uretilemedi. Yukaridaki hata iletisi incelenmelidir.')


Sabah oturumunda grafik arayüz ile gerçekleştirilen işlem, bu bölümde komut
satırı araçlarıyla ve kaba-taneli çözünürlükte tamamlanmıştır. Komut satırı
yaklaşımının belirleyici üstünlüğü, işlemin tekrarlanabilir ve raporlanabilir
olmasıdır.


---
## 8. İsteğe bağlı: kısa enerji minimizasyonu

Süre elverdiği takdirde kısa bir minimizasyon çalıştırılarak sistemin kararlı
olduğu doğrulanabilir.


In [ ]:
!gmx mdrun -deffnm em -nsteps 200 -v 2>&1 | tail -20


---
## Sık karşılaşılan hata iletileri

| Hata iletisi | Nedeni | Çözümü |
|---|---|---|
| `Atomtype X not found` | Kuvvet alanı parametre dosyası eksik | 4. bölümdeki indirmeler denetlenmelidir |
| `number of coordinates does not match topology` | `[ molecules ]` bölümündeki sayılar hatalı | 6. bölüm yeniden çalıştırılmalıdır |
| `Unknown molecule type Protein` | Protein adı topoloji dosyalarında farklı | 6. bölüm bu düzeltmeyi otomatik yapmaktadır |
| `mkdssp not found` | DSSP kurulu değil | 3. bölümdeki `-ss` seçeneği kullanılmalıdır |
| `System has non-zero total charge` | Yuvarlama kaynaklı; olağandır | `-maxwarn` ile geçilebilir |
| LINCS uyarısı veya sistem kararsızlığı | Yerleşimde çakışma bulunmaktadır | `insane` kutu boyutu büyütülmelidir |

---

## Kaynaklar

- [Martini Protein Model — Using Martinize2](https://cgmartini.nl/docs/tutorials/Martini3/ProteinsI/Tut1.html)
- [Modeling Complex Lipid Membranes — INSANE](https://cgmartini.nl/docs/tutorials/Martini3/LipidsII/)
- [Notes and Limitations](https://cgmartini.nl/docs/tutorials/Martini3/ProteinsI/Tut4.html)
- Kuvvet alanı dosyaları: [marrink-lab/martini-forcefields](https://github.com/marrink-lab/martini-forcefields)


---
## Çıktıların indirilmesi

Colab çalışma zamanı sonlandığında üretilen dosyalar silinmektedir. Aşağıdaki
hücre tüm çıktıları tek bir arşiv dosyası hâlinde indirmektedir.


In [ ]:
!zip -q -r oturum4_ciktilar.zip at2r.pdb at2r_cg.pdb molecule_*.itp sistem.gro sistem.top minimization.mdp em.tpr 2>/dev/null
from google.colab import files
files.download('oturum4_ciktilar.zip')
